### RAG(Retrieval-Augmented Generation), 검색 증강 생성
- 생성형 AI가 학습하지 않은 최신 정보나 특정 조직의 내부 데이터를 바탕으로 답변할 수 있도록 돕는 기술이다.
- R (Retrieval): 질문과 관련된 문서를 방대한 데이터베이스에서 검색한다.
- A (Augmentation): 검색된 정보를 질문과 결합하여 프롬프트를 보강한다.
- G (Generation): 보강된 맥락을 바탕으로 최종 답변을 생성한다.

### RAG 개발 순서
1. 문서 불러오기
2. 문서 분할하기
3. 임베딩
4. Vector DB 생성 및 저장
5. 검색기 생성
6. 프롬프트 생성
7. LLM 생성
8. 체인 실행

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
from langchain_community.document_loaders import TextLoader

# 텍스트 로더 생성
loader = TextLoader("경로")

# 문서 로드
docs = loader.load()

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

# CSV 로더 생성
loader = CSVLoader(file_path="경로")

# 데이터 로드
docs = loader.load()

In [ ]:
from langchain_community.document_loaders import UnstructuredExcelLoader

# Excel 로더 생성
loader = UnstructuredExcelLoader("경로", mode="elements")

# 문서 로드
docs = loader.load()

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

# 1단계, 문서 로드
FILE_PATH = "./documents/ai커리큘럼.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()

print(f"문서의 페이지 수: {len(docs)}")

In [ ]:
print(docs[0].page_content)

In [ ]:
docs[0].__dict__

In [ ]:
# 2단계, 문서 분할

# chunk_size
# LLM의 기억력과 검색의 정확도를 결정짓는다.
# 너무 작으면 맥락이 잘려서 LLM의 이해도가 떨어지고
# 너무 크면 불필요한 정보가 많이 섞여서 답변의 초점이 흐려진다.
# 500~800자: 3~4문단

# chunk_overlap
# 단락을 분할하면서 생기는 정보의 손실을 최소화하기 위해 연결고리를 만들어준다.
# 피자는 느끼해 난 그래서 싫어
# chunk1: 피자는 느끼해 난 그래서
# chunk2: 느끼해 난 그래서 싫어
# overlap: 느끼해 난 그래서
# 보통 chunk_size의 10~20%정도가 적당하다고 판단한다.

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크(조각)의 수: {len(split_documents)}")

In [ ]:
# 3단계, 임베딩
embeddings = OpenAIEmbeddings()

In [ ]:
# 4단계, 벡터 DB
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

In [ ]:
vectorstore.similarity_search("PyTorch")[0].page_content

In [ ]:
# 5단계, 검색기 생성

# 1. k=1로 설정: 아주 핵심적인 정보 하나만 딱 본다.
# retriever_1 = vectorstore.as_retriever(search_kwargs={"k": 1})
# 2. k=5로 설정: 주변 맥락까지 폭넓게 읽어서 풍부하게 답변한다.
# retriever_5 = vectorstore.as_retriever(search_kwargs={"k": 5})
# 3. 기본값: k=4
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
retriever.invoke("프로젝트")

In [ ]:
#  6단계, 프롬프트 생성
prompt_template = PromptTemplate.from_template(
    """
    귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

    #Context: 
    {context}
    
    #Question:
    {question}
    
    #Answer:
    """
)

In [ ]:
# 7단계, LLM 생성
llm = ChatOpenAI(
    model_name="gpt-5.4-nano",
    temperature=0
)

In [ ]:
# 8단계, 체인 생성

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

In [ ]:
question = "LangChain 체인 설계는 몇 주차에 배우는가?"
answer = chain.invoke(question)
print(answer)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [2]:
# docker run -d --name redis-stack -p 6380:6379 -p 8001:8001 redis/redis-stack-server:latest
# docker exec -it redis-stack redis-cli
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisSemanticCache
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. 캐시용 임베딩 모델
cache_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-nli")

# 2. 시맨틱 캐시 설정 (Docker로 띄운 Redis Stack 연결)
set_llm_cache(RedisSemanticCache(
    redis_url="redis://localhost:6380",
    embedding=cache_embeddings,
    score_threshold=0.1  # RAG는 답변의 정확도가 중요하므로 임계값을 보수적으로(낮게) 설정
))

# 단계 1: 문서 로드(Load Documents)
FILE_PATH = "./documents/ai커리큘럼.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()
print(f"문서의 페이지수: {len(docs)}")

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크의수: {len(split_documents)}")

# 단계 3: 임베딩(Embedding) 생성
# 우리 컴퓨터 내에서 모델을 돌리므로 데이터가 외부로 나가지 않음
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli", # 한국어 성능이 검증된 오픈소스 모델
    model_kwargs={'device': 'cpu'}    # GPU가 있다면 'cuda'
)

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어 생성
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성한다.
# 1. k=1로 설정: 아주 핵심적인 정보 하나만 딱 본다.
# retriever_1 = vectorstore.as_retriever(search_kwargs={"k": 1})
# 2. k=5로 설정: 주변 맥락까지 폭넓게 읽어서 풍부하게 답변한다.
# retriever_5 = vectorstore.as_retriever(search_kwargs={"k": 5})
# 3. 기본값: k=4
retriever = vectorstore.as_retriever()

# 단계 6: 프롬프트 생성(Create Prompt)
prompt = PromptTemplate.from_template(
    """귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

#Context: 
{context}

#Question:
{question}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
llm = ChatOpenAI(model_name="gpt-5.4-mini", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "LangChain 체인 설계는 몇 주차에 배우는가?"
response = chain.invoke(question)
print(response)

C:\Users\sokko\anaconda3\envs\llmplz\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


문서의 페이지수: 3
분할된 청크의수: 5
LangChain 체인 설계는 **10주차**에 배웁니다.


In [3]:
question = "LangChain 체인 설계는 몇 주차에 배우는가?"
response = chain.invoke(question)
print(response)

LangChain 체인 설계는 **10주차**에 배웁니다.
